# Author New Trial

Write a model in the notebook, validate it, try it locally, package it to `model.py`, and run it through the standard runner.

In [ ]:
from __future__ import annotations

import copy
import os
from dataclasses import asdict
from pathlib import Path

from IPython.display import display

import automl
from automl import data, eval, trial
from automl.model import BaseModel, validate_model
from automl.runner import run_trial

DRY_RUN = True
NAMESPACE = os.getenv("AUTOML_NOTEBOOK_NAMESPACE", "")
BASE_SLUG = "notebook_baseline"
RUN_TRIAL = os.getenv("AUTOML_E2E_NOTEBOOKS") == "1"


In [ ]:
active = automl.use_project(dry_run=DRY_RUN, namespace=NAMESPACE)
config = active.config
display(
    {
        "project": active.project_name,
        "repo_root": str(config.repo_root),
        "project_dir": str(config.project_dir),
        "experiment": active.active_experiment_id,
        "dry_run": active.dry_run,
        "namespace": active.namespace or "<none>",
    }
)

loaded = data.materialize(session=active)
run_config = active.config.require_run_config()
train = data.load_dataset(split_name=run_config.train_split, session=active)
holdout = data.load_dataset(split_name=run_config.eval_split, session=active)
loaded.dataset.id


In [ ]:
class NotebookModel(BaseModel):
    name = "notebook_model"

    def fit(self, df_train, registry, seed=0):
        from sklearn.compose import ColumnTransformer
        from sklearn.dummy import DummyClassifier
        from sklearn.impute import SimpleImputer
        from sklearn.pipeline import Pipeline

        target = registry.get_by_flag("target")[0]
        required_entries = self.required_transformer_entries()
        required_columns = [
            col for _, _, cols in required_entries for col in cols
        ]
        numeric_columns = [
            col
            for col in registry.get_by_flag("feature")
            if col not in {target, "SPLIT_PCT", *required_columns}
            and col in df_train.select_dtypes(include="number").columns
        ]
        self.feature_cols = [*required_columns, *numeric_columns]
        self.preprocessor = ColumnTransformer(
            [
                *required_entries,
                ("numeric", Pipeline([("imputer", SimpleImputer())]), numeric_columns),
            ]
        )
        self.model = DummyClassifier(strategy="prior", random_state=seed)
        X = self.preprocessor.fit_transform(df_train, df_train[target])
        self.model.fit(X, df_train[target])
        self.feature_registry = copy.deepcopy(registry)
        self.feature_registry.set_flag(self.feature_registry.get_by_flag("feature"), "model", False)
        for col in self.feature_cols:
            self.feature_registry.set_flag(col, "model", True)

    def transform(self, df):
        return self.preprocessor.transform(df)

    def _predict(self, X):
        return self.model.predict_proba(X)[:, 1]


In [ ]:
validate_model(NotebookModel, df=train.df, registry=train.registry, session=active)


In [ ]:
model = NotebookModel()
model.fit(train.df, train.registry)
target = train.registry.get_by_flag("target")[0]
pred = model.predict(model_input=holdout.df.drop(columns=[target]))
eval.evaluate_frame(
    y_pred=pred,
    df=holdout.df,
    spec=active.config.require_eval_spec(),
    target_col=target,
    session=active,
)


In [ ]:
model_path = trial.package_model(
    NotebookModel,
    imports=["import copy", "from automl.model import BaseModel"],
    output_path=Path("scratch/notebook_model.py"),
)
model_path


In [ ]:
# Manual Hack 
# RUN_TRIAL = True

In [ ]:
draft_dir = None
draft_slug = None
slug_index = 1
while draft_dir is None:
    candidate_slug = BASE_SLUG if slug_index == 1 else f"{BASE_SLUG}_{slug_index}"
    try:
        draft_dir = trial.create(
            candidate_slug,
            "human_baseline",
            hypothesis="Notebook-authored baseline.",
            training_origin="human",
            model_source=model_path,
            session=active,
        )
        draft_slug = candidate_slug
    except FileExistsError:
        slug_index += 1

run_result = None
if RUN_TRIAL:
    run_result = run_trial(draft_dir, session=active)
    if run_result.status != "FINISHED":
        raise RuntimeError(run_result.error or f"trial run failed: {run_result.status}")

{
    "draft_slug": draft_slug,
    "draft_dir": str(draft_dir),
    "model_path": str(draft_dir / "model.py"),
    "run_result": asdict(run_result) if run_result is not None else None,
    "run_command": f"uv run automl --project {active.project_name} trial run {draft_dir}",
}
